[Python, Visually](https://johnfisher-ai.github.io/Python-Visual-Guides/) &nbsp;&rsaquo;&nbsp; [PyMongo and Beanie, Deep Dive](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)

# Modeling Without Joins &middot; Solutions


One way to do each task. Not the only way. If yours runs and does what was asked, yours is
right too.

The first cell is the notebook's boot cell, with `COMMENT`, `size_of` and `work_done`. Run it
first. Each task opens its own client and closes it, so they can be run in any order.


In [1]:
import os
import random
import subprocess
import sys
import time
from importlib.metadata import PackageNotFoundError, version

try:
    if version("pymongo") != "4.18.1" or version("beanie") != "2.2.0":
        raise PackageNotFoundError
except PackageNotFoundError:
    subprocess.run([sys.executable, "-m", "pip", "install", "--quiet", "--root-user-action=ignore",
                    "pymongo==4.18.1", "beanie==2.2.0"], check=True)

import bson
import pymongo
from pymongo import ReturnDocument

DBPATH = "/content/mongo" if os.path.isdir("/content") else "/tmp/guide_mongo/rs"
LOGPATH = f"{DBPATH}.log"
URI = "mongodb://127.0.0.1:27017/shop"                              # no credential, anywhere
PUBLISHED = ["jammy", "noble"]                                      # codenames MongoDB builds for


def shell(command):
    """Run a shell command and hand back what it printed, without letting it stop the notebook."""
    done = subprocess.run(command, shell=True, capture_output=True, text=True)
    return done.returncode, (done.stdout + done.stderr).strip()


def answering(timeout=2000):
    """Whether a mongod is there, asked directly rather than through topology discovery."""
    try:
        with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                                 serverSelectionTimeoutMS=timeout) as client:
            client.admin.command("ping")
            return True
    except pymongo.errors.PyMongoError:
        return False


def install_server():
    """Add MongoDB's own apt repository and install the server package. Linux only."""
    if shell("which mongod")[0] == 0:
        return "already installed"

    codename = shell("lsb_release -cs")[1]
    if codename not in PUBLISHED:                                   # an unpublished one breaks apt
        print(f"  Ubuntu '{codename}' has no MongoDB repository; using '{PUBLISHED[-1]}' instead")
        codename = PUBLISHED[-1]

    if not shell("grep -o avx /proc/cpuinfo | head -1")[1]:
        raise RuntimeError("This CPU has no AVX. Every MongoDB build since 5.0 needs it, so "
                           "neither the apt package nor the tarball will start here.")

    sudo = "" if os.geteuid() == 0 else "sudo "
    shell(f"curl -fsSL https://www.mongodb.org/static/pgp/server-8.0.asc "
          f"| {sudo}gpg --dearmor -o /usr/share/keyrings/mongodb-8.0.gpg")
    shell(f'echo "deb [signed-by=/usr/share/keyrings/mongodb-8.0.gpg] '
          f'https://repo.mongodb.org/apt/ubuntu {codename}/mongodb-org/8.0 multiverse" '
          f'| {sudo}tee /etc/apt/sources.list.d/mongodb-8.0.list')
    shell(f"{sudo}apt-get -qq update "                              # this one list file only
          f"-o Dir::Etc::sourcelist=sources.list.d/mongodb-8.0.list "
          f"-o Dir::Etc::sourceparts=-")
    code, out = shell(f"{sudo}apt-get -qq -y install mongodb-org-server")
    if shell("which mongod")[0] != 0:
        raise RuntimeError(f"mongodb-org-server did not install. apt said: {out[-400:]}")
    return f"installed from the {codename} repository"


def start_server(wait=30):
    """Start mongod with a replica set name, idempotently. Returns what it had to do."""
    if answering():
        return "already running"
    if sys.platform != "linux":
        raise RuntimeError("No mongod is answering on 127.0.0.1:27017. Start your own server "
                           "with --replSet rs0 and run this again: this cell only installs one "
                           "on Linux, which is what Colab runs.")

    print(" ", install_server())
    os.makedirs(DBPATH, exist_ok=True)
    code, out = shell(f"mongod --dbpath {DBPATH} --replSet rs0 --bind_ip 127.0.0.1 "
                      f"--fork --logpath {LOGPATH}")
    if code != 0:                                                   # --fork hides the reason
        print("  mongod did not start. The last lines of its log:")
        print("   ", shell(f"tail -20 {LOGPATH}")[1].replace("\n", "\n    "))
        raise RuntimeError("mongod exited. The log above says why.")

    for attempt in range(1, wait + 1):
        if answering():
            return "installed and started"
        print(f"  waiting for mongod ({attempt})")
        time.sleep(1)
    raise RuntimeError(f"mongod did not answer within {wait} seconds.")

def initiate(wait=30):
    """Make the single node a replica set, which is what transactions and migrations need."""
    with pymongo.MongoClient("mongodb://127.0.0.1:27017/?directConnection=true",
                             serverSelectionTimeoutMS=2000) as boot:
        try:                                                        # an explicit host, not getHostName()
            boot.admin.command("replSetInitiate",
                               {"_id": "rs0", "members": [{"_id": 0, "host": "127.0.0.1:27017"}]})
        except pymongo.errors.OperationFailure as error:
            if error.code != 23:                                    # 23 is AlreadyInitialized
                raise

        for attempt in range(1, wait + 1):
            hello = boot.admin.command("hello")
            if hello.get("isWritablePrimary"):
                return f"replica set {hello['setName']}, primary"
            time.sleep(1)
    raise RuntimeError(f"No primary after {wait} seconds. The last hello was: {hello}")

SIZE = 500                                                          # Indexes and the catalog raise this
KINDS = ["laptop", "monitor", "keyboard", "mouse", "cable"]
MAKERS = ["Aster", "Belden", "Corvid", "Dalgo"]


def seed(size=None, force=False):
    """Fill shop.products and shop.reviews, once, from a fixed seed so every run agrees."""
    size = SIZE if size is None else size
    client = pymongo.MongoClient(URI, tz_aware=True)
    shop = client.get_default_database()

    if not force and shop.products.estimated_document_count() == size:
        client.close()
        return size

    shop.products.drop()
    shop.reviews.drop()
    random.seed(0)                                                  # the whole reason runs agree

    products, reviews = [], []
    for number in range(size):
        kind = KINDS[number % len(KINDS)]
        product = {
            "_id": number,
            "sku": f"{kind[:3].upper()}-{number:06d}",
            "name": f"{MAKERS[number % len(MAKERS)]} {kind} {number}",
            "maker": MAKERS[number % len(MAKERS)],
            "kind": kind,
            "price": round(random.uniform(5, 2000), 2),
            "stock": random.randint(0, 400),
            "tags": sorted(random.sample(["sale", "new", "refurbished", "bulk", "clearance"], 2)),
            "size": {"w": random.randint(5, 60), "h": random.randint(2, 40)},
        }
        products.append(product)
        for _ in range(random.randint(0, 3)):
            reviews.append({"product_id": number, "stars": random.randint(1, 5),
                            "body": f"A review of {product['name']}"})

    for start in range(0, len(products), 5000):                     # batches, not one huge insert
        shop.products.insert_many(products[start:start + 5000])
    for start in range(0, len(reviews), 5000):
        shop.reviews.insert_many(reviews[start:start + 5000])

    client.close()
    return size


def report():
    """One line naming what this notebook is running against."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        build = client.admin.command("buildInfo")["version"].split(".")[0]
        shop = client.get_default_database()
        return (f"MongoDB {build} | pymongo {version('pymongo')} | beanie {version('beanie')} "
                f"| products: {shop.products.count_documents({})}")

COMMENT = {"by": "someone", "body": "x" * 900}                      # about a kilobyte each


def failed(error):
    """A failure's real message, without the parts that change every run. The server reports a
    size in hex as well as in decimal, and a hex number is not worth committing to a notebook."""
    details = getattr(error, "details", None) or {}
    message = details.get("errmsg", str(error).split(", full error")[0])
    message = message.split(" :: caused by :: ")[-1]
    while "(0x" in message:
        start = message.index("(0x")
        message = message[:start] + message[message.index(")", start) + 1:]
    return f"{type(error).__name__}: {message}"


def size_of(document):
    """How many bytes this document takes as BSON, which is what the limit counts."""
    return len(bson.encode(document))


def work_done(pipeline, collection="products"):
    """What the server read to run a pipeline, which is how a $lookup is judged."""
    with pymongo.MongoClient(URI, tz_aware=True) as client:
        shop = client.get_default_database()
        explained = shop.command("explain",
                                 {"aggregate": collection, "pipeline": pipeline, "cursor": {}},
                                 verbosity="executionStats")
        stats = explained.get("executionStats")
        if stats is None:
            stats = explained["stages"][0]["$cursor"]["executionStats"]
        return {"documents": stats["documentsExamined"] if "documentsExamined" in stats
                else stats["totalDocsExamined"],
                "index keys": stats["totalKeysExamined"]}


print("server: ", start_server())
print("replica:", initiate())
print("seeded: ", seed(), "products and", end=" ")
with pymongo.MongoClient(URI, tz_aware=True) as _client:
    print(_client.get_default_database().reviews.count_documents({}), "reviews")
print(report())


server:  already running
replica: replica set rs0, primary
seeded:  500 products and 767 reviews
MongoDB 8 | pymongo 4.18.1 | beanie 2.2.0 | products: 500


**1.** How much fits.


In [2]:
for count in (1_000, 8_000, 16_000, 17_000):
    document = {"_id": 1, "comments": [dict(COMMENT) for _ in range(count)]}
    over = "over" if size_of(document) > 16_777_216 else "under"
    print(f"  {count:6} comments: {size_of(document):>11,} bytes  ({over} the limit)")


    1000 comments:     936,919 bytes  (under the limit)
    8000 comments:   7,502,919 bytes  (under the limit)
   16000 comments:  15,012,919 bytes  (under the limit)
   17000 comments:  15,951,919 bytes  (under the limit)


About sixteen thousand kilobyte comments, and then the document cannot be written at all. That
number is the whole design decision.


**2.** Over the wall.


In [3]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
shop.answers.drop()

try:
    shop.answers.insert_one({"_id": 1, "comments": [dict(COMMENT) for _ in range(20_000)]})
except pymongo.errors.DocumentTooLarge as error:
    print(f"{type(error).__module__}.{type(error).__name__}")
    print("  ", error)

print("written:", shop.answers.count_documents({}))
client.close()


pymongo.errors.DocumentTooLarge
   BSON document too large (18769139 bytes) - the connected server supports BSON document sizes up to 16793598 bytes.
written: 0


Nothing was written and nothing reached the server. PyMongo encoded the document, measured it, and
refused.


**3.** A product and its reviews, in one query.


In [4]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

print(list(shop.products.aggregate([
    {"$match": {"_id": 7}},
    {"$lookup": {"from": "reviews", "localField": "_id",
                 "foreignField": "product_id", "as": "reviews"}},
    {"$project": {"_id": 0, "name": 1, "stars": "$reviews.stars"}},
])))
client.close()


[{'name': 'Dalgo keyboard 7', 'stars': []}]


`as` always produces an array, even when there is one match or none, so the projection pulls the
field out of it with `"$reviews.stars"`.


**4.** What the index is worth.


In [5]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()

pipeline = [{"$match": {"kind": "monitor"}},
            {"$lookup": {"from": "reviews", "localField": "_id",
                         "foreignField": "product_id", "as": "reviews"}}]

shop.reviews.drop_indexes()
print("without an index:", work_done(pipeline))
shop.reviews.create_index("product_id", name="product_id_1")
print("with one:        ", work_done(pipeline))
client.close()


without an index: {'documents': 867, 'index keys': 100}
with one:         {'documents': 267, 'index keys': 267}


Without the index the server reads every review to build its lookup table. With one it reads only
the reviews that match, and the gap grows with the size of the foreign collection rather than with
the size of the answer.


**5.** Both shapes, side by side.


In [6]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
shop.answers.drop()
shop.answer_lines.drop()

lines = [{"sku": "KEY-000007", "qty": 2}, {"sku": "MOU-000003", "qty": 1}]

shop.answers.insert_one({"_id": "embedded", "lines": lines})
shop.answers.insert_one({"_id": "referenced"})
shop.answer_lines.insert_many([{**line, "order_id": "referenced"} for line in lines])

print("embedded:  ", shop.answers.find_one({"_id": "embedded"}))
print("referenced:", shop.answers.find_one({"_id": "referenced"}),
      list(shop.answer_lines.find({"order_id": "referenced"}, {"_id": 0, "order_id": 0})))
client.close()


embedded:   {'_id': 'embedded', 'lines': [{'sku': 'KEY-000007', 'qty': 2}, {'sku': 'MOU-000003', 'qty': 1}]}
referenced: {'_id': 'referenced'} [{'sku': 'KEY-000007', 'qty': 2}, {'sku': 'MOU-000003', 'qty': 1}]


Two lines is bounded and read with the order, so embedded is right here. The referenced version
needs a second query and an index on `order_id` to be worth anything.


**6.** Two copies, one change.


In [7]:
client = pymongo.MongoClient(URI, tz_aware=True)
shop = client.get_default_database()
shop.answers.drop()

original = shop.products.find_one({"_id": 11}, {"name": 1})
shop.answers.insert_one({"_id": 1, "product_id": 11, "product_name": original["name"]})

shop.products.update_one({"_id": 11}, {"$set": {"name": "renamed"}})
print("the product:", shop.products.find_one({"_id": 11}, {"_id": 0, "name": 1}))
print("the copy:   ", shop.answers.find_one({"_id": 1}, {"_id": 0, "product_name": 1}))

shop.products.update_one({"_id": 11}, {"$set": {"name": original["name"]}})
client.close()


the product: {'name': 'renamed'}
the copy:    {'product_name': 'Dalgo monitor 11'}


No error, no warning, and two answers to one question. A copy is only safe when it is meant to
record what was true at a particular moment, and the field name should say so.


---

&#8592; **Back to:** [Modeling Without Joins](https://colab.research.google.com/github/johnfisher-ai/Python-Visual-Guides/blob/main/notebooks/pymongo-and-beanie-deep-dive/10-modeling-without-joins.ipynb)  &nbsp;&middot;&nbsp;  [PyMongo and Beanie, Deep Dive Notebooks](https://johnfisher-ai.github.io/Python-Visual-Guides/pymongo-and-beanie-deep-dive.html)
